# Sentiment Analysis Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: a real mini-dataset

In [ ]:
```python

POSITIVE = [

    "absolutely loved this movie",

    "beautiful cinematography and a great story",

    "one of the best films of the year",

    "brilliant acting from the lead",

    "heartwarming and funny",

]

NEGATIVE = [

    "boring and far too long",

    "not worth your time",

    "the plot made no sense",

    "terrible acting, awful script",

    "i want my two hours back",

]

In [ ]:
```

Small on purpose. Real work uses tens of thousands of examples (IMDb, SST-2, Yelp polarity). The math is identical.

### Step 2: multinomial Naive Bayes from scratch

In [ ]:
```python

import math

from collections import Counter

def train_nb(docs_by_class, vocab, alpha=1.0):

    class_priors = {}

    class_word_probs = {}

    total_docs = sum(len(d) for d in docs_by_class.values())

    for cls, docs in docs_by_class.items():

        class_priors[cls] = len(docs) / total_docs

        counts = Counter()

        for doc in docs:

            for token in doc:

                counts[token] += 1

        total = sum(counts.values()) + alpha * len(vocab)

        class_word_probs[cls] = {

            w: (counts[w] + alpha) / total for w in vocab

        }

    return class_priors, class_word_probs

def predict_nb(doc, class_priors, class_word_probs):

    scores = {}

    for cls in class_priors:

        s = math.log(class_priors[cls])

        for token in doc:

            if token in class_word_probs[cls]:

                s += math.log(class_word_probs[cls][token])

        scores[cls] = s

    return max(scores, key=scores.get)

In [ ]:
```

Additive smoothing (alpha=1.0) is Laplace smoothing. Without it, a word unseen in a class has probability zero and the log explodes. `alpha=0.01` is common in practice. `alpha=1.0` is the teaching default.

### Step 3: logistic regression from scratch

In [ ]:
```python

import numpy as np

def sigmoid(x):

    return 1.0 / (1.0 + np.exp(-np.clip(x, -20, 20)))

def train_lr(X, y, epochs=500, lr=0.05, l2=0.01):

    n_features = X.shape[1]

    w = np.zeros(n_features)

    b = 0.0

    for _ in range(epochs):

        logits = X @ w + b

        preds = sigmoid(logits)

        err = preds - y

        grad_w = X.T @ err / len(y) + l2 * w

        grad_b = err.mean()

        w -= lr * grad_w

        b -= lr * grad_b

    return w, b

def predict_lr(X, w, b):

    return (sigmoid(X @ w + b) >= 0.5).astype(int)

In [ ]:
```

L2 regularization matters here. Text features are sparse; without L2 the model memorizes training examples. Start at `0.01` and tune.

### Step 4: handling negation (the failure mode)

Consider "not good" and "not bad". A BoW classifier sees `{not, good}` and `{not, bad}` and learns from whichever showed up more in training. A bigram classifier sees `not_good` and `not_bad` and learns them as distinct features. That is usually enough.

A cruder fix that works when you do not have bigrams: **negation scoping**. Prefix tokens following a negation word with `NOT_` up to the next punctuation.

In [ ]:
```python

NEGATION_WORDS = {"not", "no", "never", "nor", "none", "nothing", "neither"}

NEGATION_TERMINATORS = {".", "!", "?", ",", ";"}

def apply_negation(tokens):

    out = []

    negate = False

    for token in tokens:

        if token in NEGATION_TERMINATORS:

            negate = False

            out.append(token)

            continue

        if token in NEGATION_WORDS:

            negate = True

            out.append(token)

            continue

        out.append(f"NOT_{token}" if negate else token)

    return out

In [ ]:
```

In [ ]:
```python

>>> apply_negation(["not", "good", "at", "all", ".", "but", "funny"])

['not', 'NOT_good', 'NOT_at', 'NOT_all', '.', 'but', 'funny']

In [ ]:
```

Now `good` and `NOT_good` are different features. The classifier can weight them opposite. Three lines of preprocessing, measurable accuracy jump on sentiment benchmarks.

### Step 5: evaluation metrics that matter

Accuracy alone is misleading if classes are imbalanced. Real sentiment corpora are usually 70-80% positive or 70-80% negative; a constant-majority classifier gets 80% accuracy and is worthless. Report every one of the following:

- **Per-class precision and recall.** One pair per class. Macro-average them to get a single number that respects class balance.

- **Macro-F1 (primary metric for imbalanced data).** Mean of per-class F1 scores, equally weighted. Use this instead of accuracy when classes are imbalanced.

- **Weighted-F1 (alternative).** Same as macro but weighted by class frequency. Report alongside macro-F1 when the imbalance itself has business meaning.

- **Confusion matrix.** Raw counts. Always inspect before trusting any scalar metric; it reveals which pair of classes the model confuses.

- **Per-class error samples.** Pull 5 wrong predictions per class. Read them. Nothing replaces reading the actual errors.

For severely imbalanced data (> 95-5 ratio), report **AUROC** and **AUPRC** instead of accuracy. AUPRC is more sensitive to the minority class, which is what you usually care about (spam, fraud, rare sentiment).

**Common bug to avoid.** Reporting micro-F1 instead of macro-F1 on imbalanced data gives a number that looks high because it is dominated by the majority class. Macro-F1 forces you to see the minority-class performance.

In [ ]:
```python

def evaluate(y_true, y_pred):

    tp = sum(1 for t, p in zip(y_true, y_pred) if t == 1 and p == 1)

    fp = sum(1 for t, p in zip(y_true, y_pred) if t == 0 and p == 1)

    fn = sum(1 for t, p in zip(y_true, y_pred) if t == 1 and p == 0)

    tn = sum(1 for t, p in zip(y_true, y_pred) if t == 0 and p == 0)

    precision = tp / (tp + fp) if tp + fp else 0

    recall = tp / (tp + fn) if tp + fn else 0

    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0

    return {"tp": tp, "fp": fp, "tn": tn, "fn": fn, "precision": precision, "recall": recall, "f1": f1}

In [ ]:
```

## Exercises

In [ ]:
1. **Easy.** Add `apply_negation` as a preprocessing step in the scikit-learn pipeline and measure the F1 delta on a small sentiment dataset.
2. **Medium.** Implement class-weighted logistic regression (pass `class_weight="balanced"` to scikit-learn, or derive the gradient yourself). Measure the effect on a synthetic 90-10 class imbalance.
3. **Hard.** Build a sarcasm detector by training a second classifier on the residuals of the sentiment model. Document your experimental setup. Warn the reader when your accuracy is below chance (chance-level on 2-class sarcasm is ~50%, and most first attempts land there).